In [109]:
from pathlib import Path
import gcamreader
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

In [110]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4': 28,  # Methane
    'CH4_AGR': 28,  # Methane from Agriculture
    'CH4_AWB': 28,  # Methane from Agricultural Waste Burning
    'N2O': 265,  # Nitrous Oxide
    'N2O_AGR': 265,  # Nitrous Oxide from Agriculture
    'N2O_AWB': 265,  # Nitrous Oxide from Agricultural Waste Burning
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [111]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [112]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250717"
# dbfile = "database_basexdb_test"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Current-Policy, Enhanced-Ambition


In [113]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Current-Policy', 'Enhanced-Ambition']

In [114]:
scenarios.reverse()

In [115]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [206]:
i = 109
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df.head()

cement production by tech and vintage


,Units,scenario,region,sector,subsector,technology,output,Year,value
0,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2025",cement,2025,0.003135
1,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2025",cement,2030,0.002725
2,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2025",cement,2035,0.002044
3,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2030",cement,2030,0.010838
4,Mt,Current-Policy,South Korea,cement,cement,"cement CCS,year=2030",cement,2035,0.009424


In [214]:
df['tech'] = df['technology'].str.split(',').str[0]
dfFig = df[(df['Year'] >= 2005)].groupby(['scenario', 'Year', 'tech'])['value'].sum().reset_index()
dfFig.head()

,scenario,Year,tech,value
0,Current-Policy,2005,cement,53.82570
1,Current-Policy,2010,cement,52.50110
2,Current-Policy,2015,cement,52.50110
3,Current-Policy,2020,cement,46.36590
4,Current-Policy,2025,cement,42.07977


In [215]:
# stack_order = [
#     'BF', 'BF-CCS', 'BF-H2', 'BF-Biomass', 'EAF-scrap', 'DRI-EAF', 'DRI-EAF-CCS', 'DRI-EAF-H2', 
# ]
stack_order = [
    'cement', 'cement LC3', 'cement CCS'
]

In [216]:
df['tech'] = pd.Categorical(df['tech'], categories=stack_order, ordered=True)
df = df.sort_values(by=['Year', 'tech'])
df['tech'].unique()

['cement', 'cement LC3', 'cement CCS']
Categories (3, object): ['cement' < 'cement LC3' < 'cement CCS']

In [217]:
dfAgg1 = dfFig[(dfFig['scenario'] == 'Current-Policy')]
fig1 = px.bar(dfAgg1, x="Year", y="value", color="tech", title="Current Policy")
dfAgg2 = dfFig[(dfFig['scenario'] == 'Enhanced-Ambition')]
fig2 = px.bar(dfAgg2, x="Year", y="value", color="tech", title="Enhanced Ambition")

In [218]:
# ---- Colour & hatch (“pattern”) settings ------------------------
fuel_colors = {
    "cement LC3": "#FFB785",    # peach
    "cement" : "grey",   # vivid red
    "cement CCS"    : "#0057B5",   # deep blue
}

# optional: hatch / pattern overlay for the categories that are
# drawn with diagonal stripes in the figure
fuel_patterns = {

}

# Example of applying in Plotly
import plotly.graph_objects as go

def bar_for(fuel, x, y):
    return go.Bar(
        name   = fuel,
        x      = x,
        y      = y,
        marker = dict(
            color   = fuel_colors[fuel],
            pattern = dict(shape = fuel_patterns.get(fuel, ""))
        )
    )

In [219]:
years = list(range(2005, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# ------------------------------------------------------------------
# add traces from the first figure → left pane
# ------------------------------------------------------------------
for tr in fig1.data:
    tr.showlegend = False                      # keep legend single
    # NEW → colour & pattern injection
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            # marker.pattern is available from Plotly 5.3+
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=1, secondary_y=False)

# ------------------------------------------------------------------
# add traces from the second figure → right pane
# ------------------------------------------------------------------
for tr in fig2.data:
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=2, secondary_y=False)


fig.update_layout(
    yaxis=dict(title="EJ", showgrid=True),
    yaxis1=dict(title="EJ", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 60]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)


fig.update_xaxes(tickangle=45)

fig.update_yaxes(range=[None, 60])

fig.update_layout(
    yaxis=dict(title="Mtoe", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Chemical Feedstock Inputs</b>",
        font=dict(size=28),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=20),
        x=1.02, y=1,
        borderwidth=0
    )
)
fig.update_xaxes(
    tickvals=years,
    ticktext=[str(y) for y in years]
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
fig.update_layout(title=dict(font=dict(size=28)))
fig

In [155]:
i = 100
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df#.head()

industry final energy by tech and fuel


,Units,scenario,region,sector,subsector,technology,input,Year,value
0,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2025,0.000060
1,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2030,0.000279
2,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2035,0.000594
3,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2025,0.000055
4,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2030,0.000201
...,...,...,...,...,...,...,...,...,...
1245,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2015,0.002158
1246,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2020,0.002368
1247,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2025,0.002479
1248,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2030,0.003229


In [156]:
df['sector'].unique()

array(['agricultural energy use', 'ammonia', 'cement',
       'chemical energy use', 'chemical feedstocks',
       'construction energy use', 'construction feedstocks',
       'food processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'paper', 'process heat cement', 'process heat food processing',
       'process heat paper', 'waste biomass for paper', 'CO2 removal',
       'process heat dac'], dtype=object)

In [194]:
dfFig = df[(df['Year'] >= 2005) & (df['sector'] == 'chemical feedstocks')].copy()

In [195]:
dfFig['input'].unique()

array(['delivered coal', 'refined liquids industrial',
       'delivered biomass'], dtype=object)

In [196]:
def cat_fuel(fuel):

    if fuel in ['delivered coal']:
        return 'Coal'
    elif fuel == 'refined liquids industrial':
        return 'Oil'
    elif fuel in ['delivered biomass']:
        return 'Biomass'

In [197]:
dfFig['fuel'] = dfFig['input'].apply(cat_fuel)

In [198]:
stack_order = [
    'Oil', 'Biomass', 'Coal'
]

In [199]:
dfFig['fuel'] = pd.Categorical(dfFig['fuel'], categories=stack_order, ordered=True)
dfFig = dfFig.sort_values(by=['Year', 'fuel'])
dfFig['fuel'].unique()

['Oil', 'Coal', 'Biomass']
Categories (3, object): ['Oil' < 'Biomass' < 'Coal']

In [200]:
dfFig['value'] *= 23.8846

In [201]:
dfAgg1 = dfFig[(dfFig['scenario'] == 'Current-Policy')]
fig1 = px.bar(dfAgg1, x="Year", y="value", color="fuel", title="Current Policy")
dfAgg2 = dfFig[(dfFig['scenario'] == 'Enhanced-Ambition')]
fig2 = px.bar(dfAgg2, x="Year", y="value", color="fuel", title="Enhanced Ambition")

In [202]:
# ---- Colour & hatch (“pattern”) settings ------------------------
fuel_colors = {
    "Oil" : "#EF0C0C",   # vivid red
    "Coal"           : "#000000",   # solid black
    "Biomass"        : "#099B43",   # medium green
}

# optional: hatch / pattern overlay for the categories that are
# drawn with diagonal stripes in the figure
fuel_patterns = {

}

# Example of applying in Plotly
import plotly.graph_objects as go

def bar_for(fuel, x, y):
    return go.Bar(
        name   = fuel,
        x      = x,
        y      = y,
        marker = dict(
            color   = fuel_colors[fuel],
            pattern = dict(shape = fuel_patterns.get(fuel, ""))
        )
    )

In [204]:
years = list(range(2005, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# ------------------------------------------------------------------
# add traces from the first figure → left pane
# ------------------------------------------------------------------
for tr in fig1.data:
    tr.showlegend = False                      # keep legend single
    # NEW → colour & pattern injection
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            # marker.pattern is available from Plotly 5.3+
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=1, secondary_y=False)

# ------------------------------------------------------------------
# add traces from the second figure → right pane
# ------------------------------------------------------------------
for tr in fig2.data:
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=2, secondary_y=False)


fig.update_layout(
    yaxis=dict(title="EJ", showgrid=True),
    yaxis1=dict(title="EJ", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 60]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)


fig.update_xaxes(tickangle=45)

fig.update_yaxes(range=[None, 60])

fig.update_layout(
    yaxis=dict(title="Mtoe", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Chemical Feedstock Inputs</b>",
        font=dict(size=28),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=20),
        x=1.02, y=1,
        borderwidth=0
    )
)
fig.update_xaxes(
    tickvals=years,
    ticktext=[str(y) for y in years]
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
fig.update_layout(title=dict(font=dict(size=28)))
fig

In [122]:
i = 119
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

iron and steel production by tech


,Units,scenario,region,sector,subsector,technology,output,Year,value
0,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,1975,0.665514
1,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,1990,13.226000
2,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2005,26.767200
3,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2010,34.108100
4,Mt,Current-Policy,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2015,48.479100
...,...,...,...,...,...,...,...,...,...
73,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,EAF with scrap,iron and steel,2015,20.863000
74,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,EAF with scrap,iron and steel,2020,19.561710
75,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,EAF with scrap,iron and steel,2025,22.081730
76,Mt,Enhanced-Ambition,South Korea,iron and steel,EAF with scrap,EAF with scrap,iron and steel,2030,25.609220


In [123]:
df.groupby(['Year', 'scenario'])['value'].sum()

Year  scenario         
1975  Current-Policy        1.163048
      Enhanced-Ambition     1.163048
1990  Current-Policy       23.124960
      Enhanced-Ambition    23.124960
2005  Current-Policy       47.820000
      Enhanced-Ambition    47.820000
2010  Current-Policy       58.913960
      Enhanced-Ambition    58.913960
2015  Current-Policy       69.669959
      Enhanced-Ambition    69.669959
2020  Current-Policy       64.499878
      Enhanced-Ambition    64.499878
2025  Current-Policy       67.814804
      Enhanced-Ambition    68.683622
2030  Current-Policy       69.223171
      Enhanced-Ambition    61.030579
2035  Current-Policy       58.575832
      Enhanced-Ambition    59.364401
Name: value, dtype: float64

In [124]:
df['technology'].unique()

array(['BLASTFUR', 'BLASTFUR CCS', 'BLASTFUR with hydrogen',
       'Biomass-based', 'EAF with DRI', 'EAF with DRI CCS',
       'Hydrogen-based DRI', 'EAF with scrap'], dtype=object)

In [125]:
def cat_tech(tech):

    if tech == 'BLASTFUR':
        return 'BF'
    if tech in [
        'BLASTFUR', 'BLASTFUR CCS', 'BLASTFUR with hydrogen', 'Biomass-based'
    ]:
        return 'BF'
    # elif tech == 'BLASTFUR CCS':
    #     return 'BF-CCS'
    # elif tech == 'BLASTFUR with hydrogen':
    #     return 'BF-H2'
    # elif tech == 'Biomass-based':
    #     return 'BF-Biomass'
    elif tech == 'EAF with DRI':
        return 'DRI-EAF'
    elif tech == 'EAF with DRI CCS':
        return 'DRI-EAF-CCS'
    elif tech == 'Hydrogen-based DRI':
        return 'DRI-EAF-H2'
    elif tech == 'EAF with scrap':
        return 'EAF-scrap'

In [126]:
df['tech'] = df['technology'].apply(cat_tech)

In [127]:
# stack_order = [
#     'BF', 'BF-CCS', 'BF-H2', 'BF-Biomass', 'EAF-scrap', 'DRI-EAF', 'DRI-EAF-CCS', 'DRI-EAF-H2', 
# ]
stack_order = [
    'BF', 'EAF-scrap', 'DRI-EAF', 'DRI-EAF-CCS', 'DRI-EAF-H2', 
]

In [128]:
df['tech'] = pd.Categorical(df['tech'], categories=stack_order, ordered=True)
df = df.sort_values(by=['Year', 'tech'])
df['tech'].unique()

['BF', 'EAF-scrap', 'DRI-EAF', 'DRI-EAF-CCS', 'DRI-EAF-H2']
Categories (5, object): ['BF' < 'EAF-scrap' < 'DRI-EAF' < 'DRI-EAF-CCS' < 'DRI-EAF-H2']

In [129]:
dfAgg1 = df[(df['Year'] >= 2005) & (df['scenario'] == 'Current-Policy')].groupby(['Year', 'tech'])['value'].sum().reset_index()
fig1 = px.bar(dfAgg1, x="Year", y="value", color="tech", title="Current Policy")
dfAgg2 = df[(df['Year'] >= 2005) & (df['scenario'] == 'Enhanced-Ambition')].groupby(['Year', 'tech'])['value'].sum().reset_index()
fig2 = px.bar(dfAgg2, x="Year", y="value", color="tech", title="Enhanced Ambition")

/tmp/ipykernel_47447/2787647340.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_47447/2787647340.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [130]:
# ---- Colour & hatch (“pattern”) settings ------------------------
fuel_colors = {
    "BF"           : "#000000",   # solid black
    # "BF-CCS"       : "grey",   # same black, but we’ll add a hatch
    # "BF-H2"            : "#E4EAED",   # bright blue
    "EAF-scrap"        : "#492681",   # lighter cyan-blue
    # "BF-Biomass"        : "#099B43",   # medium green
    "DRI-EAF"    : "#0057B5",   # deep blue
    "DRI-EAF-H2"       : "#FED30B",   # golden yellow
    "DRI-EAF-CCS"            : "#4DDADC",   # white base; will rely on hatch only
}

# optional: hatch / pattern overlay for the categories that are
# drawn with diagonal stripes in the figure
fuel_patterns = {
    "BF-CCS": "/",   # forward-slash hatch
    "DRI-EAF-CCS" : "/",   # same
    "DRI-EAF-H2"     : "\\",  # back-slash hatch
}

# Example of applying in Plotly
import plotly.graph_objects as go

def bar_for(fuel, x, y):
    return go.Bar(
        name   = fuel,
        x      = x,
        y      = y,
        marker = dict(
            color   = fuel_colors[fuel],
            pattern = dict(shape = fuel_patterns.get(fuel, ""))
        )
    )

In [131]:
# tell px.bar your exact category order,
# so that color‐groups (and legend entries) come out in stack_order
fig1 = px.bar(
    dfAgg1,
    x="Year", y="value", color="tech",
    category_orders={'tech': stack_order},
    title="Current Policy"
)
fig2 = px.bar(
    dfAgg2,
    x="Year", y="value", color="tech",
    category_orders={'tech': stack_order},
    title="Enhanced Ambition"
)

# stack the bars, and keep legend trace order = data order
for fig in (fig1, fig2):
    fig.update_layout(
        barmode='stack',
        legend_traceorder='normal',   # ← respect the data‐order
    )

In [132]:
years = list(range(2005, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# ------------------------------------------------------------------
# add traces from the first figure → left pane
# ------------------------------------------------------------------
for tr in fig1.data:
    tr.showlegend = False                      # keep legend single
    # NEW → colour & pattern injection
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            # marker.pattern is available from Plotly 5.3+
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=1, secondary_y=False)

# ------------------------------------------------------------------
# add traces from the second figure → right pane
# ------------------------------------------------------------------
for tr in fig2.data:
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=2, secondary_y=False)


fig.update_layout(
    yaxis=dict(title="EJ", showgrid=True),
    yaxis1=dict(title="EJ", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)


fig.update_xaxes(tickangle=45)

fig.update_yaxes(range=[None, 90])

fig.update_layout(
    yaxis=dict(title="Mt", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Iron & Steel Production by Technology</b>",
        font=dict(size=33),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=20),
        x=1.02, y=1,
        borderwidth=0
    )
)
fig.update_xaxes(
    tickvals=years,
    ticktext=[str(y) for y in years]
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
fig.update_layout(title=dict(font=dict(size=28)))

# fig.show()

fig

In [133]:
i = 100
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

industry final energy by tech and fuel


,Units,scenario,region,sector,subsector,technology,input,Year,value
0,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2025,0.000060
1,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2030,0.000279
2,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2035,0.000594
3,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2025,0.000055
4,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2030,0.000201
...,...,...,...,...,...,...,...,...,...
1245,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2015,0.002158
1246,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2020,0.002368
1247,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2025,0.002479
1248,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2030,0.003229


In [134]:
def cat_fuel(row):
    fuel = row['input']
    tech = row['technology']
    if tech in ['gas CCS', 'biomass CCS', 'coal CCS', 'refined liquids CCS']:
        return tech
    elif tech in ['hightemp DAC NG', 'hightemp DAC elec', 'lowtemp DAC heatpump']:
        return 'DAC'
    elif tech == 'electricity with solar':
        return 'electricity'
    elif tech == 'gas with solar':
        return 'gas' 
    elif fuel in ['H2 wholesale dispensing', 'H2 wholesale delivery', 'H2 industrial']:
        return 'hydrogen'
    elif fuel == 'elect_td_ind':
        return 'electricity'
    elif fuel in ['refined liquids industrial']:
        return 'refined liquids'
    elif fuel in ['delivered biomass', 'regional woodpulp for energy']:
        return 'biomass'
    elif fuel in ['wholesale gas']:
        return 'gas'
    elif fuel in ['delivered coal']:
        return 'coal'
    else:
        print(fuel, tech)
        return 'others'

In [135]:
df['Units'].unique()

array(['EJ'], dtype=object)

In [136]:
df['input'].unique()

array(['elect_td_ind', 'H2 wholesale dispensing',
       'refined liquids industrial', 'delivered biomass', 'wholesale gas',
       'H2 wholesale delivery', 'H2 industrial', 'delivered coal',
       'global solar resource', 'regional woodpulp for energy'],
      dtype=object)

In [137]:
df['fuel'] = df.apply(cat_fuel, axis=1)
df

,Units,scenario,region,sector,subsector,technology,input,Year,value,fuel
0,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2025,0.000060,electricity
1,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2030,0.000279,electricity
2,EJ,Current-Policy,South Korea,agricultural energy use,mobile,electricity,elect_td_ind,2035,0.000594,electricity
3,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2025,0.000055,hydrogen
4,EJ,Current-Policy,South Korea,agricultural energy use,mobile,hydrogen,H2 wholesale dispensing,2030,0.000201,hydrogen
...,...,...,...,...,...,...,...,...,...,...
1245,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2015,0.002158,biomass
1246,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2020,0.002368,biomass
1247,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2025,0.002479,biomass
1248,EJ,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,biomass cogen,regional woodpulp for energy,2030,0.003229,biomass


In [150]:
df[(df['fuel'] == 'biomass') & (df['Year'] >= 2025) & (df['scenario'] == 'Enhanced-Ambition')].pivot(index=['sector', 'subsector', 'technology', 'input'], columns='Year', values='value')

Year                                                                                    2025  \
sector                       subsector  technology    input                                    
agricultural energy use      stationary biomass       delivered biomass             0.002823   
chemical energy use          biomass    biomass       delivered biomass             0.011573   
chemical feedstocks          biomass    biomass       delivered biomass                  NaN   
construction energy use      stationary biomass       delivered biomass             0.001197   
iron and steel               BLASTFUR   Biomass-based delivered biomass             0.012247   
other industrial energy use  biomass    biomass       delivered biomass             0.087805   
                                        biomass cogen delivered biomass             0.002487   
process heat cement          biomass    biomass       delivered biomass             0.038093   
process heat food processing biomass    biomass       delivered biomass             0.003871   
                                        biomass cogen delivered biomass             0.000001   
waste biomass for paper      biomass    biomass       regional woodpulp for energy  0.025294   
                                        biomass cogen regional woodpulp for energy  0.002479   

Year                                                                                    2030  \
sector                       subsector  technology    input                                    
agricultural energy use      stationary biomass       delivered biomass             0.001620   
chemical energy use          biomass    biomass       delivered biomass             0.011258   
chemical feedstocks          biomass    biomass       delivered biomass             0.262057   
construction energy use      stationary biomass       delivered biomass             0.001181   
iron and steel               BLASTFUR   Biomass-based delivered biomass             0.043491   
other industrial energy use  biomass    biomass       delivered biomass             0.086676   
                                        biomass cogen delivered biomass             0.002454   
process heat cement          biomass    biomass       delivered biomass             0.037126   
process heat food processing biomass    biomass       delivered biomass             0.003774   
                                        biomass cogen delivered biomass             0.000008   
waste biomass for paper      biomass    biomass       regional woodpulp for energy  0.024271   
                                        biomass cogen regional woodpulp for energy  0.003229   

Year                                                                                    2035  
sector                       subsector  technology    input                                   
agricultural energy use      stationary biomass       delivered biomass             0.000631  
chemical energy use          biomass    biomass       delivered biomass             0.010626  
chemical feedstocks          biomass    biomass       delivered biomass             0.391390  
construction energy use      stationary biomass       delivered biomass             0.001000  
iron and steel               BLASTFUR   Biomass-based delivered biomass             0.033493  
other industrial energy use  biomass    biomass       delivered biomass             0.081464  
                                        biomass cogen delivered biomass             0.002443  
process heat cement          biomass    biomass       delivered biomass             0.034520  
process heat food processing biomass    biomass       delivered biomass             0.003156  
                                        biomass cogen delivered biomass             0.000029  
waste biomass for paper      biomass    biomass       regional woodpulp for energy  0.021848  
                                        biomass cogen regional woodpulp for

In [152]:
df[(df['Year'] >= 2025) & (df['scenario'] == 'Enhanced-Ambition') & (df['sector'] == 'chemical feedstocks')]

,Units,scenario,region,sector,subsector,technology,input,Year,value,fuel
764,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,biomass,biomass,delivered biomass,2030,0.262057,biomass
765,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,biomass,biomass,delivered biomass,2035,0.391390,biomass
769,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,coal,coal,delivered coal,2025,0.015178,coal
770,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,coal,coal,delivered coal,2030,0.010183,coal
771,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,coal,coal,delivered coal,2035,0.008419,coal
778,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,refined liquids,refined liquids,refined liquids industrial,2025,1.900560,refined liquids
779,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,refined liquids,refined liquids,refined liquids industrial,2030,1.618610,refined liquids
780,EJ,Enhanced-Ambition,South Korea,chemical feedstocks,refined liquids,refined liquids,refined liquids industrial,2035,1.433460,refined liquids


In [138]:
dfFig = df[(df['Year'] >= 2005)].groupby(['scenario', 'Year', 'fuel'])['value'].sum().reset_index()
dfFig

,scenario,Year,fuel,value
0,Current-Policy,2005,biomass,0.107670
1,Current-Policy,2005,coal,0.823345
2,Current-Policy,2005,electricity,0.671997
3,Current-Policy,2005,gas,0.213206
4,Current-Policy,2005,refined liquids,1.797392
...,...,...,...,...
108,Enhanced-Ambition,2035,gas,0.630696
109,Enhanced-Ambition,2035,gas CCS,0.043384
110,Enhanced-Ambition,2035,hydrogen,0.067028
111,Enhanced-Ambition,2035,refined liquids,1.837666


In [139]:
dfFig['value'] *= 23.8846

In [140]:
# dfFig['fuel'] = dfFig['fuel'].apply(lambda fuel: fuel.capitalize())
dfFig['fuel'].unique()

array(['biomass', 'coal', 'electricity', 'gas', 'refined liquids',
       'biomass CCS', 'coal CCS', 'gas CCS', 'hydrogen',
       'refined liquids CCS', 'DAC'], dtype=object)

In [141]:
stack_order = [
    'DAC', 'hydrogen', 'electricity', 'biomass CCS', 'biomass', 'gas CCS', 'gas', 'coal CCS', 'coal', 'refined liquids CCS', 'refined liquids'
]

In [142]:
dfFig['fuel'] = pd.Categorical(dfFig['fuel'], categories=stack_order, ordered=True)
dfFig = dfFig.sort_values(by=['Year', 'fuel'])
dfFig['fuel'].unique()

['electricity', 'biomass', 'gas', 'coal', 'refined liquids', ..., 'biomass CCS', 'gas CCS', 'coal CCS', 'refined liquids CCS', 'DAC']
Length: 11
Categories (11, object): ['DAC' < 'hydrogen' < 'electricity' < 'biomass CCS' ... 'coal CCS' < 'coal' < 'refined liquids CCS' < 'refined liquids']

In [143]:
dfAgg1 = dfFig[(dfFig['scenario'] == 'Current-Policy')]
fig1 = px.bar(dfAgg1, x="Year", y="value", color="fuel", title="Current Policy")
dfAgg2 = dfFig[(dfFig['scenario'] == 'Enhanced-Ambition')]
fig2 = px.bar(dfAgg2, x="Year", y="value", color="fuel", title="Enhanced Ambition")

In [144]:
# ---- Colour & hatch (“pattern”) settings ------------------------
fuel_colors = {
    "refined liquids CCS": "#FFB785",    # peach
    "refined liquids" : "#EF0C0C",   # vivid red
    "coal"           : "#000000",   # solid black
    "coal CCS"       : "#000000",   # same black, but we’ll add a hatch
    "gas"            : "#0186E0",   # bright blue
    "gas CCS"        : "#33A8E2",   # lighter cyan-blue
    "biomass"        : "#099B43",   # medium green
    "electricity"    : "#0057B5",   # deep blue
    "hydrogen"       : "#FED30B",   # golden yellow
    "DAC"            : "#FFFFFF",   # white base; will rely on hatch only
}

# optional: hatch / pattern overlay for the categories that are
# drawn with diagonal stripes in the figure
fuel_patterns = {
    "coal CCS": "/",   # forward-slash hatch
    "gas CCS" : "/",   # same
    "refined liquids CCS": "/",
    "biomass CCS": "/",
    "DAC"     : "\\",  # back-slash hatch
}

# Example of applying in Plotly
import plotly.graph_objects as go

def bar_for(fuel, x, y):
    return go.Bar(
        name   = fuel,
        x      = x,
        y      = y,
        marker = dict(
            color   = fuel_colors[fuel],
            pattern = dict(shape = fuel_patterns.get(fuel, ""))
        )
    )

In [145]:
years = list(range(2005, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# ------------------------------------------------------------------
# add traces from the first figure → left pane
# ------------------------------------------------------------------
for tr in fig1.data:
    tr.showlegend = False                      # keep legend single
    # NEW → colour & pattern injection
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            # marker.pattern is available from Plotly 5.3+
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=1, secondary_y=False)

# ------------------------------------------------------------------
# add traces from the second figure → right pane
# ------------------------------------------------------------------
for tr in fig2.data:
    if tr.name in fuel_colors:
        tr.marker.color = fuel_colors[tr.name]
        pattern = fuel_patterns.get(tr.name)
        if pattern:
            tr.marker.pattern = dict(shape=pattern)

    fig.add_trace(tr, row=1, col=2, secondary_y=False)


fig.update_layout(
    yaxis=dict(title="EJ", showgrid=True),
    yaxis1=dict(title="EJ", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)


fig.update_xaxes(tickangle=45)

fig.update_yaxes(range=[None, 150])

fig.update_layout(
    yaxis=dict(title="Mtoe", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    title=dict(
        text="<b>Final energy consumption by fuel</b>",
        font=dict(size=28),
        x=0.5
    ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=20),
        x=1.02, y=1,
        borderwidth=0
    )
)
fig.update_xaxes(
    tickvals=years,
    ticktext=[str(y) for y in years]
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=21))

# (if you also set a main title earlier)
fig.update_layout(title=dict(font=dict(size=28)))
fig